# Đánh giá Hệ thống Acne Advisor AI bằng RAGAS (100 Kịch bản Toàn diện)

Notebook chạy đánh giá định lượng toàn diện hệ thống **Acne Advisor AI** trên bộ benchmark **100 kịch bản tiếng Việt** (bao gồm tra cứu trực tiếp, ngôn ngữ đời thường người bệnh, cầu nối biệt dược thương mại trên GraphRAG, hội thoại nối tiếp đa lượt có `conversation_history`, và từ chối/cảnh báo an toàn) theo phương pháp **RAGAS** (*Retrieval Augmented Generation Assessment*, Es et al., 2023, `arXiv:2309.15217`).

### Cấu hình chấm độc lập (tránh thiên vị tự chấm):
- **System LLM:** `gemini-3.5-flash-lite` (dự phòng `gemini-3.1-flash-lite`)
- **Judge LLM (NVIDIA NIM):** `nvidia/nemotron-3-ultra-550b-a55b` (550B / 55B active), dự phòng `nvidia/nemotron-3.5-super-vl-120b-a12b` & `mistralai/mistral-large-4-1t-a52b`
- **Embedding Model:** mặc định `models/gemini-embedding-2` (3072 chiều), tùy chọn `nvidia/nemotron-3.5-embed-8b` (4096 chiều)

### Các chỉ số được đo:
1. **Faithfulness** (*Độ trung thực với bằng chứng*): phát hiện ảo giác (hallucination).
2. **Answer Relevancy** (*Độ liên quan câu trả lời*): đo mức độ bám sát câu hỏi người dùng.
3. **Context Precision** (*Độ chính xác ngữ cảnh*): đo độ chính xác xếp hạng top ngữ cảnh truy hồi.
4. **Context Recall** (*Độ bao phủ ngữ cảnh*): đo ngữ cảnh truy hồi có đủ ý theo đáp án chuẩn.
5. **Answer Correctness** (*Độ chính xác câu trả lời*): kết hợp độ chính xác sự kiện (75%) và độ tương đồng ngữ nghĩa (25%) so với đáp án chuẩn.
6. **Safe Abstention Accuracy** (*Tỷ lệ từ chối an toàn*): kiểm tra khả năng chặn tình huống khẩn cấp / từ chối đúng ca ngoài phạm vi hoặc thiếu bằng chứng.

Kết quả chi tiết được hiển thị trực tiếp ngay bên dưới các cell và đồng thời xuất ra **`evaluation/results/ragas_evaluation_report.md`**.

In [ ]:
# 1. Khởi tạo đường dẫn dự án và nạp biến môi trường (.env)
from __future__ import annotations
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'evaluation':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

load_dotenv(PROJECT_ROOT / '.env', override=True)
print('Project root:', PROJECT_ROOT)
print('GOOGLE_API_KEY configured:', bool(os.getenv('GOOGLE_API_KEY')))
print('NVIDIA_API_KEY configured:', bool(os.getenv('NVIDIA_API_KEY')))
print('QDRANT_URL:', os.getenv('QDRANT_URL', 'http://127.0.0.1:6333'))
print('NEO4J_URI:', os.getenv('NEO4J_URI', 'bolt://127.0.0.1:7687'))

In [ ]:
# 2. Cấu hình tham số đánh giá và kiểm tra Knowledge Build hiện hành
import pandas as pd
from evaluation.ragas_runner import (
    DEFAULT_DATASET_PATH,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_FALLBACK_JUDGE_MODELS,
    DEFAULT_JUDGE_MODEL,
    DEFAULT_JUDGE_PROVIDER,
    DEFAULT_RESULTS_DIR,
    export_evaluation_artifacts,
    load_evaluation_dataset,
    render_markdown_report,
    run_system_on_dataset,
    score_records_with_ragas,
    summarize_evaluation,
)
from src.knowledge.versioning import resolve_active_knowledge_build_id

DATASET_PATH = PROJECT_ROOT / DEFAULT_DATASET_PATH
RESULTS_DIR = PROJECT_ROOT / DEFAULT_RESULTS_DIR
SYSTEM_CHECKPOINT_PATH = RESULTS_DIR / 'ragas_system_records.json'
SCORED_CHECKPOINT_PATH = RESULTS_DIR / 'ragas_scored_checkpoint.json'
SYSTEM_MODEL = os.getenv('GOOGLE_MODEL', 'gemini-3.5-flash-lite')
JUDGE_PROVIDER = os.getenv('RAGAS_JUDGE_PROVIDER', DEFAULT_JUDGE_PROVIDER)
JUDGE_MODEL = os.getenv('RAGAS_JUDGE_MODEL', DEFAULT_JUDGE_MODEL)
JUDGE_LABEL = JUDGE_MODEL if '/' in JUDGE_MODEL else f'{JUDGE_PROVIDER}/{JUDGE_MODEL}'
_raw_fallbacks = os.getenv('RAGAS_FALLBACK_JUDGE_MODELS', '')
FALLBACK_JUDGE_MODELS = (
    tuple(m.strip() for m in _raw_fallbacks.split(',') if m.strip())
    if _raw_fallbacks.strip()
    else DEFAULT_FALLBACK_JUDGE_MODELS
)
EMBEDDING_MODEL = os.getenv('RAGAS_EMBEDDING_MODEL', DEFAULT_EMBEDDING_MODEL)
MAX_CASES = None

kb_build_id = resolve_active_knowledge_build_id()
print('Active KB Build ID:', kb_build_id)
print('System Model:', SYSTEM_MODEL, '| Judge:', JUDGE_LABEL, '| Fallback Judges:', FALLBACK_JUDGE_MODELS)

In [ ]:
# 3. Nạp bộ dữ liệu đánh giá chuẩn và thống kê phân bố kịch bản
from IPython.display import display

dataset_meta, all_cases = load_evaluation_dataset(DATASET_PATH)
cases = all_cases[:MAX_CASES] if MAX_CASES else all_cases
print(
    f'Tổng số kịch bản: {len(cases)} '
    f'(Trả lời: {sum(1 for c in cases if c.expected_behavior == "answer")}, '
    f'Từ chối an toàn: {sum(1 for c in cases if c.expected_behavior == "abstain")}, '
    f'Hội thoại đa lượt: {sum(1 for c in cases if c.conversation_history)})'
)
df_overview = pd.DataFrame([
    {
        'case_id': c.case_id,
        'category': c.category,
        'dialogue_type': c.dialogue_type,
        'expected': c.expected_behavior,
        'turns': len(c.conversation_history) + 1,
        'question': c.user_input[:72] + ('...' if len(c.user_input) > 72 else ''),
    }
    for c in cases
])
display(df_overview.groupby(['category', 'dialogue_type']).size().reset_index(name='count'))
display(df_overview.head(20))

In [ ]:
# 4. Chạy hệ thống thực tế (Hybrid GraphRAG + LangGraph Agent) trên toàn bộ kịch bản
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
system_records = await run_system_on_dataset(
    cases,
    llm_provider='gemini',
    llm_model=SYSTEM_MODEL,
    allow_model_fallback=True,
    case_delay_seconds=1.2,
    checkpoint_path=SYSTEM_CHECKPOINT_PATH,
    verbose=True,
)
print(
    f'Hoàn tất chạy hệ thống: {len(system_records)} ca | '
    f'Lỗi runtime: {sum(1 for r in system_records if r.get("error"))}'
)

In [ ]:
# 5. Chấm điểm độc lập bằng RAGAS (5 chỉ số cốt lõi + Safe Abstention Accuracy)
scored_records = await score_records_with_ragas(
    system_records,
    judge_model=JUDGE_MODEL,
    judge_provider=JUDGE_PROVIDER,
    fallback_judge_models=FALLBACK_JUDGE_MODELS,
    embedding_model=EMBEDDING_MODEL,
    checkpoint_path=SCORED_CHECKPOINT_PATH,
    verbose=True,
)
summary = summarize_evaluation(
    scored_records,
    dataset_meta=dataset_meta,
    judge_model=JUDGE_LABEL,
    embedding_model=EMBEDDING_MODEL,
    kb_build_id=kb_build_id,
)
df_summary = pd.DataFrame([
    {'metric': k, **v} for k, v in summary['ragas_summary'].items()
])
display(df_summary)
print('Safe Abstention:', summary['safe_abstention'])
print('Counts:', summary['counts'])
print('Latency (ms):', summary['latency_ms'])
print('Token Usage:', summary['token_usage'])

In [ ]:
# 6. Xuất báo cáo chi tiết ra file Markdown (.md), JSON thô và hiển thị tổng quan
from IPython.display import Markdown, display

report_path, raw_path = export_evaluation_artifacts(
    summary,
    output_dir=RESULTS_DIR,
    report_filename='ragas_evaluation_report.md',
    raw_filename='ragas_evaluation_raw.json',
)
print('Đã xuất báo cáo Markdown:', report_path)
print('Đã xuất dữ liệu thô JSON:', raw_path)
report_md = render_markdown_report(summary)
preview_md = report_md.split('## 6. Chi tiết câu trả lời')[0].strip()
display(Markdown(preview_md))